# IE Teachers Knowledge Graph Notebook
This Colab-first notebook walks through parsing IE teacher biographies, extracting entities, normalising labels, and building a NetworkX knowledge graph with friendly QA checkpoints.


In [1]:
import sys
sys.path.append('/content/ie-teachers-kg')

In [2]:
# Install dependencies (Restart & Run-All safe)
!pip install -q -r /content/ie-teachers-kg/requirements.txt

In [3]:
# Global imports and deterministic setup
import os
import sys
import random
import json
from datetime import datetime
from collections import Counter

import numpy as np

random.seed(42)
np.random.seed(42)

REPO = "/content/ie-teachers-kg"
if REPO not in sys.path:
    sys.path.append(REPO)
SRC_PATH = os.path.join(REPO, "src")
if os.path.isdir(SRC_PATH) and SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

os.makedirs(os.path.join(REPO, "outputs"), exist_ok=True)
print("Repo path:", REPO)
print("Output dir:", os.path.join(REPO, "outputs"))


Repo path: /content/ie-teachers-kg
Output dir: /content/ie-teachers-kg/outputs


## Section A — Setup
We install the lightweight NLP stack (transformers + spaCy + NetworkX) and register the repo on `sys.path` so `src/` helpers are importable inside Colab.


## Section B — Load Data


In [4]:
import pandas as pd

DATA_PATH = os.path.join(REPO, "data", "teachers_db_practice.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} professor bios from {DATA_PATH}")
df[['alias', 'area', 'position', 'full_info']].head()


Loaded 1,228 professor bios from /content/ie-teachers-kg/data/teachers_db_practice.csv


,alias,area,position,full_info
0,Appius Aemilius Agricola,Architecture & Design,NaN,<p> has worked as a designer for the last dec...
1,Appius Aemilius Cicero,Economics,NaN,<p>Mr. Madgar has been teaching economics par...
2,Appius Aemilius Crassus,Private & Business Law,NaN,<p>Lawyer with broad experience in Market Regu...
3,Flavia Prisca,Economics,Adjunct professor,<p> is a seasoned leader with a proven track r...
4,Appius Aemilius Scipio,Science & Technology,NaN,"<p> Carrio is a seasoned technology leader, re..."


## Section C — Sectioning, NER, and rule-based extraction
We clean HTML, segment the bios into logical sections, run two multilingual NER models, merge their outputs, and enrich the detections with regex rules (degrees, years, courses).


In [7]:

from tqdm.auto import tqdm

from parsing import (
    clean_html,
    split_sections,
    iter_bullets,
    parse_corporate_bullet,
    parse_academic_background_bullet,
    parse_academic_experience_bullet,
)
from ner_hf import load_pipelines, ner_on_line
from fusion import fuse_line
from normalize import normalize_name
from rules import SECTION_STOP_ORGS, ORG_ALIASES, LOCATION_ALIASES, canon_org, canon_location

pipes = load_pipelines()
print("Loaded pipelines:", list(pipes.keys()))


ImportError: attempted relative import with no known parent package

In [ ]:
import re
from typing import Callable, Dict, List, Optional

SECTION_PARSERS: Dict[str, Callable[[str], Optional[dict]]] = {
    "corporate_experience": parse_corporate_bullet,
    "academic_experience": parse_academic_experience_bullet,
    "academic_background": parse_academic_background_bullet,
}
SECTION_ORDER = ["corporate_experience", "academic_experience", "academic_background"]

COURSE_PATTERNS = [
    re.compile(r"Adjunct (?:Level [^ ]+ )?Professor of (?P<course>[^,]+)", re.IGNORECASE),
    re.compile(r"Professor of (?P<course>[^,]+)", re.IGNORECASE),
    re.compile(r"taught (?P<course>.+?) at", re.IGNORECASE),
]


def clean_course_name(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    cleaned = text.replace("“", """).replace("”", """).replace("’", "'")
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" '.,;:-")
    if not cleaned:
        return None
    tokens = cleaned.split()
    if not (1 <= len(tokens) <= 8):
        return None
    if len(tokens) == 1 and tokens[0].lower() not in {"economics"}:
        return None
    if not any(ch.isalpha() for ch in cleaned):
        return None
    return cleaned


def course_from_line(line: str, parsed_course: Optional[str]) -> Optional[str]:
    for candidate in [parsed_course]:
        cleaned = clean_course_name(candidate)
        if cleaned:
            return cleaned
    for pattern in COURSE_PATTERNS:
        match = pattern.search(line)
        if match:
            cleaned = clean_course_name(match.group("course"))
            if cleaned:
                return cleaned
    return None


def valid_org(name: Optional[str]) -> bool:
    if not name:
        return False
    stripped = name.strip()
    if not stripped or stripped.lower() in SECTION_STOP_ORGS:
        return False
    return sum(ch.isalnum() for ch in stripped) >= 3


In [ ]:

extraction_records = []
university_names = []
company_names = []
location_names = []
all_courses = []

for idx, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
    html = getattr(row, 'full_info', '') or ''
    sections = split_sections(html)
    if not sections:
        sections = {'intro': clean_html(html)}
    prof_id = getattr(row, 'alias', f'prof_{idx}')
    record = {
        'prof_id': prof_id,
        'area': getattr(row, 'area', None),
        'position': getattr(row, 'position', None),
        'raw_sections': sections,
        'studies': [],
        'work': [],
        'courses': [],
    }

    for section_name in SECTION_ORDER:
        section_text = sections.get(section_name, '')
        if not section_text:
            continue
        for line in iter_bullets(section_text):
            parser = SECTION_PARSERS.get(section_name)
            parsed = parser(line) if parser else None
            ner = ner_on_line(line, pipes)
            candidates = fuse_line(section_name, line, parsed, ner)
            for cand in candidates:
                org_label = cand.org_canon or cand.org_raw
                if not valid_org(org_label):
                    continue
                location_label = cand.location_canon or cand.location_raw
                norm_location = normalize_name(location_label) if location_label else None
                if norm_location:
                    location_names.append(norm_location)
                location_canon = canon_location(location_label) if location_label else None
                base_entry = {
                    'org_canon': cand.org_canon,
                    'location': cand.location_raw,
                    'location_canon': location_canon,
                    'source_section': section_name,
                    'text_span': line,
                    'meta': cand.meta,
                    'sources': cand.meta.get('sources'),
                }
                if cand.relation == 'worked_at':
                    entry = {
                        **base_entry,
                        'company': org_label,
                        'role': cand.role,
                        'start_year': cand.start_year,
                        'end_year': cand.end_year,
                        'year_bin': cand.year_bin,
                    }
                    record['work'].append(entry)
                    company_names.append(org_label)
                elif cand.relation == 'studied_at':
                    entry = {
                        **base_entry,
                        'university': org_label,
                        'degree': cand.degree_text,
                        'degree_level': cand.degree_level,
                        'field': cand.field,
                        'year': cand.year,
                        'year_bin': cand.year_bin,
                    }
                    record['studies'].append(entry)
                    university_names.append(org_label)
                elif cand.relation == 'teaches':
                    course_name = course_from_line(line, cand.course)
                    if not course_name:
                        continue
                    center_label = org_label
                    center_norm = normalize_name(center_label) if center_label else None
                    center_canon = canon_org(center_norm or center_label)
                    entry = {
                        **base_entry,
                        'course': course_name,
                        'center': center_label,
                        'center_norm': center_norm,
                        'center_canon': center_canon or None,
                        'start_year': cand.start_year,
                        'end_year': cand.end_year,
                    }
                    record['courses'].append(entry)
                    all_courses.append(course_name)
    extraction_records.append(record)

print(f"Stored {len(extraction_records)} extraction records")


## Section D — Normalisation & Canonical labels
We clean organisation and location names, cluster near-duplicates with RapidFuzz, and map degrees to a controlled vocabulary.


In [ ]:

from normalize import normalize_name, cluster_and_canonicalize
from rules import canon_org, canon_location

CANON_DEGREES = {
    'PHD': 'PhD',
    'DOCTOR': 'PhD',
    'MSC': 'MSc',
    'MS': 'MSc',
    'MA': 'MA',
    'MBA': 'MBA',
    'BSC': 'BSc',
    'BA': 'BA',
    'MENG': 'MEng',
    'BENG': 'BEng',
}

uni_map = cluster_and_canonicalize(university_names, stopwords=SECTION_STOP_ORGS, alias_map=ORG_ALIASES) if university_names else {}
comp_map = cluster_and_canonicalize(company_names, stopwords=SECTION_STOP_ORGS, alias_map=ORG_ALIASES) if company_names else {}
loc_map = cluster_and_canonicalize(
    location_names,
    stopwords=SECTION_STOP_ORGS,
    alias_map=LOCATION_ALIASES,
    entity='location',
) if location_names else {}
org_map = {**comp_map, **uni_map}

print("University clusters:", json.dumps(uni_map, indent=2)[:500])
print("Company clusters:", json.dumps(comp_map, indent=2)[:500])
print("Location clusters:", json.dumps(loc_map, indent=2)[:500])


def canonical_degree(name):
    if not name:
        return None
    key = name.upper().replace('.', '')
    return CANON_DEGREES.get(key)


def canonical_location(value):
    if not value:
        return None
    norm = normalize_name(value)
    mapped = loc_map.get(norm, norm)
    canon = canon_location(mapped)
    return canon or None


def is_valid_course(name):
    if not name:
        return False
    stripped = name.strip(" '")
    if len(stripped) < 2:
        return False
    return any(ch.isalnum() for ch in stripped)


for record in extraction_records:
    cleaned_studies = []
    for study in record['studies']:
        uni_candidate = study.get('university_canon') or study.get('university')
        uni_norm = normalize_name(uni_candidate) if uni_candidate else None
        cluster_value = uni_map.get(uni_norm, uni_norm) if uni_norm else None
        study['university_norm'] = uni_norm
        final_uni = canon_org(cluster_value or uni_candidate)
        study['university_canon'] = final_uni or None
        if study.get('location') or study.get('location_canon'):
            study['location_canon'] = canonical_location(study.get('location_canon') or study.get('location'))
        study['degree_canon'] = study.get('degree_level') or canonical_degree(study.get('degree') or study.get('degree_text'))
        if not valid_org(study.get('university_canon')):
            continue
        cleaned_studies.append(study)
    record['studies'] = cleaned_studies

    cleaned_work = []
    for work in record['work']:
        comp_candidate = work.get('company_canon') or work.get('company')
        comp_norm = normalize_name(comp_candidate) if comp_candidate else None
        cluster_value = comp_map.get(comp_norm, comp_norm) if comp_norm else None
        work['company_norm'] = comp_norm
        final_company = canon_org(cluster_value or comp_candidate)
        work['company_canon'] = final_company or None
        if work.get('location') or work.get('location_canon'):
            work['location_canon'] = canonical_location(work.get('location_canon') or work.get('location'))
        if not valid_org(work.get('company_canon')):
            continue
        cleaned_work.append(work)
    record['work'] = cleaned_work

    cleaned_courses = []
    for course in record['courses']:
        center_source = course.get('center_canon') or course.get('center') or course.get('university') or course.get('company')
        center_norm = normalize_name(center_source) if center_source else None
        cluster_value = org_map.get(center_norm, center_norm) if center_norm else None
        course['center_norm'] = center_norm
        final_center = canon_org(cluster_value or center_source)
        course['center_canon'] = final_center or None
        if course.get('location') or course.get('location_canon'):
            course['location_canon'] = canonical_location(course.get('location_canon') or course.get('location'))
        if not is_valid_course(course.get('course')):
            continue
        cleaned_courses.append(course)
    record['courses'] = cleaned_courses


In [ ]:

# Enforce org + country-first location canon everywhere (post-processing)
from normalize import normalize_name
from rules import canon_org, canon_location

for record in extraction_records:
    # studies
    for study in record.get('studies', []):
        if study.get('university') or study.get('university_canon'):
            base = study.get('university_canon') or study.get('university')
            study['university_norm'] = normalize_name(base)
            study['university_canon'] = canon_org(study['university_norm'])
        if study.get('location') or study.get('location_canon'):
            loc = study.get('location_canon') or study.get('location')
            study['location_canon'] = canon_location(loc)

    # work
    for work in record.get('work', []):
        if work.get('company') or work.get('company_canon'):
            base = work.get('company_canon') or work.get('company')
            work['company_norm'] = normalize_name(base)
            work['company_canon'] = canon_org(work['company_norm'])
        if work.get('location') or work.get('location_canon'):
            loc = work.get('location_canon') or work.get('location')
            work['location_canon'] = canon_location(loc)

    # courses
    for course in record.get('courses', []):
        center = course.get('center_canon') or course.get('center') or course.get('university') or course.get('company')
        if center:
            course['center_norm'] = normalize_name(center)
            course['center_canon'] = canon_org(course['center_norm'])
        if course.get('location') or course.get('location_canon'):
            loc = course.get('location_canon') or course.get('location')
            course['location_canon'] = canon_location(loc)

    # misc location attachments
    for loc_item in record.get('locations', []):
        if loc_item.get('location') or loc_item.get('location_canon'):
            loc = loc_item.get('location_canon') or loc_item.get('location')
            loc_item['location_canon'] = canon_location(loc)


In [ ]:

# Acceptance tests for courses, degrees, and canonicalization
from rules import canon_org

madgar = next(rec for rec in extraction_records if rec['prof_id'] == 'Appius Aemilius Cicero')


def _find_course_by_center(rec, center_query):
    target = canon_org(center_query)
    for course in rec.get('courses', []):
        cand = course.get('center_canon') or course.get('center') or course.get('university') or course.get('company') or ''
        if canon_org(cand) == target:
            return course
    return None


ie_course = _find_course_by_center(madgar, 'IE Business School')
assert ie_course['course'] == 'Economics'
assert ie_course.get('location_canon') == 'Spain'
assert ie_course.get('start_year') == 2018
assert (ie_course.get('meta') or {}).get('end_year_text', '').lower() == 'present'

kent_course = _find_course_by_center(madgar, 'Kent State University')
assert kent_course['course'] == 'Economics'
assert kent_course.get('location_canon') == 'USA'
assert kent_course.get('start_year') == 2005
assert kent_course.get('end_year') == 2010

ie_mba = next(s for s in madgar['studies'] if canon_org(s.get('university_canon') or s.get('university')) == canon_org('IE Business School'))
brown_mba = next(s for s in madgar['studies'] if 'Brown University' in (s.get('university_canon') or s.get('university', '')))
assert ie_mba.get('degree_canon') == brown_mba.get('degree_canon') == 'MBA'
assert ie_mba.get('year') == brown_mba.get('year') == 2018
assert ie_mba.get('location_canon') == 'Spain'
assert brown_mba.get('location_canon') == 'USA'

uni_names = {s.get('university_canon') for rec in extraction_records for s in rec['studies'] if s.get('university_canon')}
assert not {u.lower() for u in uni_names if u}.intersection(SECTION_STOP_ORGS)

assert loc_map.get(normalize_name('Spaing')) == 'Spain'
assert uni_map.get(normalize_name('U. de Navarra')) == 'Universidad de Navarra'
assert uni_map.get(normalize_name('University of Navarra')) == 'Universidad de Navarra'
assert uni_map.get(normalize_name('MBA IE')) == 'IE Business School'

print('Acceptance checks passed for courses, degrees, and canonicalization.')


In [ ]:
import os
import networkx as nx

gexf_path = os.path.join(REPO, 'outputs', 'graph.gexf')
assert os.path.exists(gexf_path), 'GEXF file missing'
nx.read_gexf(gexf_path)
print('GEXF file is present and readable:', gexf_path)


## Section E — Graph building & artefacts


In [ ]:
from graph_utils import (
    new_graph,
    add_professor,
    add_university,
    add_company,
    add_course,
    add_degree,
    add_location,
    link_studied_at,
    link_worked_at,
    link_teaches,
    link_located_in,
    save_graph,
    top_k_by_degree,
)
import networkx as nx

G = new_graph()

for record in extraction_records:
    prof_node = add_professor(G, record['prof_id'], area=record.get('area'), position=record.get('position'))
    for study in record['studies']:
        univ = study.get('university_canon') or study.get('university_norm')
        if not univ:
            continue
        univ_node = add_university(G, univ, location=study.get('location_canon') or study.get('location'))
        if study.get('degree_canon'):
            add_degree(G, study['degree_canon'], field=study.get('field'))
        if study.get('location_canon'):
            link_located_in(G, univ_node, study['location_canon'], 'university')
        link_studied_at(
            G,
            prof_node,
            univ_node,
            degree=study.get('degree_canon') or study.get('degree_level') or study.get('degree'),
            field=study.get('field'),
            year=study.get('year'),
            year_bin=study.get('year_bin'),
            source_section=study.get('source_section', 'unknown'),
            text_span=study.get('text_span'),
            meta=study.get('meta'),
        )
    for work in record['work']:
        comp = work.get('company_canon') or work.get('company_norm')
        if not comp:
            continue
        comp_node = add_company(G, comp, location=work.get('location_canon') or work.get('location'))
        if work.get('location_canon'):
            link_located_in(G, comp_node, work['location_canon'], 'company')
        link_worked_at(
            G,
            prof_node,
            comp_node,
            role=work.get('role'),
            start_year=work.get('start_year'),
            end_year=work.get('end_year'),
            year_bin=work.get('year_bin'),
            source_section=work.get('source_section', 'unknown'),
            text_span=work.get('text_span'),
            meta=work.get('meta'),
        )
    for course in record['courses']:
        name = course.get('course')
        if not name:
            continue
        course_node = add_course(G, name)
        link_teaches(
            G,
            prof_node,
            course_node,
            center=course.get('center_canon') or course.get('center'),
            location=course.get('location_canon') or course.get('location'),
            start_year=course.get('start_year'),
            end_year=course.get('end_year'),
            source_section=course.get('source_section', 'unknown'),
            text_span=course.get('text_span'),
            meta=course.get('meta'),
        )

out_dir = os.path.join(REPO, 'outputs')
paths = save_graph(G, out_dir)
print('Export paths:', paths)
gexf_path = paths.get('gexf')
if not gexf_path or not os.path.exists(gexf_path):
    raise FileNotFoundError(f'GEXF missing; details: {paths}')
nx.read_gexf(gexf_path)
print('GEXF load OK.')
print(nx.info(G))
print("Top universities:", top_k_by_degree(G, 'University'))
print("Top companies:", top_k_by_degree(G, 'Company'))


In [ ]:
import random
sampled = random.sample(extraction_records, min(10, len(extraction_records)))
for item in sampled:
    print(json.dumps(item, indent=2)[:1000])
    print('-' * 80)


## Section F — Quick visualisation


In [ ]:
import random
import networkx as nx
import matplotlib.pyplot as plt

def draw_sample_graph(graph, k: int = 40, seed: int = 42):
    random.seed(seed)
    professors = [n for n, data in graph.nodes(data=True) if data.get('type') == 'Professor']
    if not professors:
        print('No professor nodes to visualize.')
        return
    sample = professors[:k] if len(professors) <= k else random.sample(professors, k)
    nodes = set(sample)
    for prof in sample:
        nodes.update(graph.neighbors(prof))
    subgraph = graph.subgraph(nodes).copy()
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(subgraph, seed=seed)
    colors = []
    palette = {
        'Professor': '#1f77b4',
        'University': '#ff7f0e',
        'Company': '#2ca02c',
        'Course': '#d62728',
        'Location': '#9467bd',
        'Degree': '#8c564b',
    }
    for node in subgraph:
        node_type = graph.nodes[node].get('type')
        colors.append(palette.get(node_type, '#7f7f7f'))
    nx.draw(subgraph, pos, with_labels=False, node_color=colors, node_size=120)
    plt.title(f'Mini knowledge subgraph (K={len(sample)} professors)')
    plt.show()

draw_sample_graph(G, k=40, seed=42)


## Section G — Export ZIP deliverable


In [ ]:
import shutil
import tempfile

stamp = datetime.utcnow().strftime('%Y%m%d')
zip_base = os.path.join(REPO, f'ie-teachers-kg_submit_{stamp}')
with tempfile.TemporaryDirectory() as tmp:
    targets = [
        ('notebooks', 'main.ipynb'),
        ('data', 'teachers_db_practice.csv'),
        ('outputs', 'nodes.csv'),
        ('outputs', 'edges.csv'),
        ('outputs', 'graph.gexf'),
        ('outputs', 'graph.graphml'),
        ('', 'requirements.txt'),
        ('', 'README.md'),
    ]
    for folder, fname in targets:
        src = os.path.join(REPO, folder, fname) if folder else os.path.join(REPO, fname)
        if os.path.exists(src):
            dst_dir = os.path.join(tmp, folder) if folder else tmp
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, fname))
    shutil.make_archive(zip_base, 'zip', tmp)

print(f"Created archive: {zip_base}.zip")


In [ ]:
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as err:
    print("Download hint: run this cell in Colab to download the ZIP.")
    print(err)


## Section H — Documentation


**Pipeline pseudocode**
```
load CSV → iterate rows
  clean HTML → split sections
  run both NER models → merge spans
  attach regex degrees/courses + nearest locations
  normalise org/location strings via RapidFuzz clusters
  populate NetworkX graph with nodes + edges + provenance
persist nodes/edges/gexf → QA prints → build Colab ZIP deliverable
```


In [ ]:
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

deg = Counter(
    (s.get('degree_canon') or s.get('degree'))
    for rec in extraction_records
    for s in rec.get('studies', [])
    if (s.get('degree_canon') or s.get('degree'))
)

unis = Counter(
    s.get('university_canon')
    for rec in extraction_records
    for s in rec.get('studies', [])
    if s.get('university_canon')
)

comps = Counter(
    w.get('company_canon')
    for rec in extraction_records
    for w in rec.get('work', [])
    if w.get('company_canon')
)

locs = Counter(
    entry.get('location_canon')
    for rec in extraction_records
    for entry in (rec.get('studies', []) + rec.get('work', []) + rec.get('courses', []))
    if entry.get('location_canon')
)

courses_counter = Counter(
    c.get('course')
    for rec in extraction_records
    for c in rec.get('courses', [])
    if c.get('course')
)

years = Counter(
    s.get('year')
    for rec in extraction_records
    for s in rec.get('studies', [])
    if isinstance(s.get('year'), int)
)

summary = {
    'Top Degrees': deg.most_common(10),
    'Top Universities': unis.most_common(10),
    'Top Companies': comps.most_common(10),
    'Top Locations': locs.most_common(10),
    'Top Courses': courses_counter.most_common(10),
    'Top Years': years.most_common(10),
}

for label, values in summary.items():
    print(label, '→', values)

def to_df(counter: Counter, column: str):
    items = counter.most_common(20)
    return pd.DataFrame(items, columns=[column, 'count']) if items else pd.DataFrame(columns=[column, 'count'])

for counter_obj, column_name in [
    (deg, 'degree'),
    (unis, 'university'),
    (comps, 'company'),
    (locs, 'location'),
    (courses_counter, 'course'),
    (years, 'year'),
]:
    display(to_df(counter_obj, column_name))

if years:
    ordered_years = sorted(years.items())
    plt.figure(figsize=(10, 4))
    plt.bar([str(year) for year, _ in ordered_years], [count for _, count in ordered_years])
    plt.xticks(rotation=90)
    plt.title('Year distribution (studies)')
    plt.tight_layout()
    plt.show()


In [ ]:
from rules import canon_org, canon_location
import networkx as nx

assert canon_org('MBA IE') == 'IE Business School'
assert canon_org('U. de Navarra') == 'Universidad de Navarra'
assert canon_location('Chestertown MD USA') == 'USA'

ie_courses = [
    course
    for rec in extraction_records
    for course in rec.get('courses', [])
    if course.get('center_canon') == 'IE Business School'
]
assert ie_courses, 'Expected IE Business School courses to be canonicalized.'

gexf_path = paths.get('gexf')
assert gexf_path and os.path.exists(gexf_path), f'GEXF missing; details: {paths}'
nx.read_gexf(gexf_path)
print('Acceptance checks passed.')
